# Posner spatial cueing: analysis

Reads one run folder (`*_trials.csv`) and computes the measures of Posner (1980), Fig. 2 (p. 7):
mean simple RT on **valid**, **neutral** and **invalid** trials, the **benefit** (neutral − valid) and the
**cost** (invalid − neutral). It also reports what a simple-RT design has to report for its RTs to be
trusted: false alarms on catch trials, misses, anticipations and, for the eye-tracker set, attempts lost
to fixation breaks.

Set `RUN` to a run folder; by default the newest run under `data/` or `data-rehearsal/` is used.

A simulated run (`--mode simulate`) shows **no** cueing effect: its observer ignores the cue on purpose
(src/posner1980/simulate.py). Only a session with a person produces data to compare with the paper.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from posner1980 import analysis

HERE = Path.cwd() if (Path.cwd() / 'run.py').exists() else Path.cwd().parent
RUN = os.environ.get('ANALYSIS_RUN')
if RUN is None:
    runs = sorted(HERE.glob('data*/**/*_trials.csv'), key=lambda p: p.stat().st_mtime)
    RUN = runs[-1].parent if runs else None
print('run:', RUN)

In [ ]:
trials = analysis.load_trials(RUN)
print('parameter set:', ', '.join(sorted(trials['fixation_control'].dropna().unique())))
print('attempts:', len(trials))
trials['outcome'].value_counts()

## Can the RTs be trusted?

Catch trials show the cue and no target; a key press there is a false alarm. A high false-alarm rate means
the subject was responding to the clock rather than to the target. RTs under 100 ms are counted as fast
guesses and left out of the means (an assumed bound, docs/methods.md).

In [ ]:
summary = analysis.response_summary(trials)
for k, v in summary.items():
    print(f'{k:>18}: {v:.3f}' if isinstance(v, float) else f'{k:>18}: {v}')
print({'false_alarm_rate': round(summary['false_alarm_rate'], 3)})

In [ ]:
if (trials['outcome'] == 'FIXATION_BREAK').any():
    print('fixation breaks by trial period (each attempt was served again):')
    print(analysis.fixation_breaks_by_period(trials).to_string())
else:
    print('no fixation breaks recorded (keyboard-only set, or none occurred)')

## Costs and benefits (Fig. 2)

In [ ]:
table = analysis.rt_by_condition(trials)
display(table.round(1))
cb = analysis.costs_and_benefits(trials)
print({k: round(v, 1) for k, v in cb.items()})
print('benefit_ms = neutral - valid; cost_ms = invalid - neutral')

In [ ]:
# Posner (1980), Fig. 2, simple RT at 7 deg: roughly 300, 260 and 245 ms for invalid,
# neutral and valid, read from the figure by eye (the paper prints no table).
paper = {'invalid': 300, 'neutral': 260, 'valid': 245}
order = ['invalid', 'neutral', 'valid']
labels = ['20%\nInvalid', '50%\nNeutral', '80%\nValid']
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(range(3), [table.loc[c, 'mean'] for c in order], 'o-', color='k', label='this run')
ax.plot(range(3), [paper[c] for c in order], 'o--', color='0.6', label='Posner (1980) Fig. 2, read by eye')
ax.set_xticks(range(3), labels)
ax.set_xlabel('Position uncertainty')
ax.set_ylabel('RT (ms)')
ax.legend(frameon=False)
plt.tight_layout()

## Per side

The paper pools the two sides. A large left/right asymmetry is worth knowing about before pooling.

In [ ]:
rts = analysis.detection_rts(trials)
rts.pivot_table(index='validity', columns='target', values='rt_ms', aggfunc='mean').reindex(list(analysis.CONDITIONS)).round(1)